# 03 — Feature engineering

## Objective

Create transparent, auditable features from the main application table while handling known data-quality issues safely.

The output is a modelling-ready dataset. All transformations are deterministic and are documented below.


## Business lens

Feature engineering converts application information into indicators that are easier to relate to affordability, stability, and credit history. Every feature must have a clear interpretation and remain traceable to the original source data.

| Feature family | Business interpretation | Example |
| --- | --- | --- |
| Affordability | Compares credit obligations with repayment capacity. | `ANNUITY_INCOME_RATIO` |
| Employment stability | Captures tenure while isolating known source-system anomalies. | `YEARS_EMPLOYED`, `EMPLOYMENT_ANOMALY` |
| Household profile | Adds context on dependants and family composition. | `CHILDREN_RATIO` |
| Credit-history proxy | Summarises available document and social-circle signals. | `NUM_DOCUMENTS`, `SOCIAL_CIRCLE_RISK` |

> Features are risk indicators, not automatic reasons to approve or reject an applicant. Their use requires validation, fairness assessment, and policy oversight.


In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "Raw"
PROCESSED_DATA_DIR = DATA_DIR / "Processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

train = pd.read_csv(RAW_DATA_DIR / "application_train.csv")
print(f"Loaded {train.shape[0]:,} applications and {train.shape[1]:,} columns.")


Loaded 307,511 applications and 122 columns.


## Data cleaning and anomaly handling

In [2]:
train = train.replace([np.inf, -np.inf], np.nan).drop_duplicates().copy()

# In this dataset, 365243 is a sentinel rather than a valid employment duration.
train["EMPLOYMENT_ANOMALY"] = (train["DAYS_EMPLOYED"] == 365243).astype("int8")
train["DAYS_EMPLOYED"] = train["DAYS_EMPLOYED"].replace(365243, np.nan)

def safe_ratio(numerator, denominator):
    return numerator.div(denominator.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)


## Financial, demographic, and household features

In [3]:
train["CREDIT_INCOME_RATIO"] = safe_ratio(train["AMT_CREDIT"], train["AMT_INCOME_TOTAL"])
train["ANNUITY_INCOME_RATIO"] = safe_ratio(train["AMT_ANNUITY"], train["AMT_INCOME_TOTAL"])
train["CREDIT_ANNUITY_RATIO"] = safe_ratio(train["AMT_CREDIT"], train["AMT_ANNUITY"])
train["PAYMENT_RATE"] = safe_ratio(train["AMT_ANNUITY"], train["AMT_CREDIT"])

train["AGE"] = (-train["DAYS_BIRTH"] / 365.25).clip(lower=0)
train["AGE_SQ"] = train["AGE"] ** 2
train["YEARS_EMPLOYED"] = (-train["DAYS_EMPLOYED"] / 365.25).clip(lower=0)
train["CHILDREN_RATIO"] = safe_ratio(train["CNT_CHILDREN"], train["CNT_FAM_MEMBERS"])


## Credit-history proxy features

In [4]:
document_columns = [column for column in train.columns if column.startswith("FLAG_DOCUMENT")]
train["NUM_DOCUMENTS"] = train[document_columns].sum(axis=1)
train["NO_DOCUMENTS"] = (train["NUM_DOCUMENTS"] == 0).astype("int8")

social_columns = [
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
]
train["SOCIAL_CIRCLE_RISK"] = train[social_columns].sum(axis=1, min_count=1)


## Validation and export

In [5]:
numeric_features = train.select_dtypes(include="number")
infinite_values = np.isinf(numeric_features.to_numpy()).sum()

feature_audit = pd.DataFrame(
    {
        "metric": ["Rows", "Columns", "Numeric columns", "Infinite numeric values"],
        "value": [len(train), train.shape[1], numeric_features.shape[1], int(infinite_values)],
    }
)
display(feature_audit)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
train.to_csv(PROCESSED_DATA_DIR / "train_feature_engineered.csv", index=False)
pd.DataFrame({"feature": train.columns}).to_csv(REPORTS_DIR / "feature_catalog.csv", index=False)

print("Feature-engineered dataset and feature catalog saved successfully.")


,metric,value
0,Rows,307511
1,Columns,134
2,Numeric columns,118
3,Infinite numeric values,0


Feature-engineered dataset and feature catalog saved successfully.


## Takeaways

- Ratio features use safe division, so invalid denominators become missing values rather than infinite values.
- The employment sentinel is explicitly flagged and converted to missing before duration is calculated.
- Missing values are intentionally retained at this stage; the modelling pipeline imputes them using training data only.


## Business hand-off

**Output for the next stage:** a governed feature set with a feature catalog that can support model validation and stakeholder review.

**Control point:** missing values are retained intentionally. Imputation happens inside the training pipeline, preventing information from the test set from influencing the model-development process.
